In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# ALEJANDRO MELGUIZO
# DATE: 8/18/2026
# TOPIC: new file to filter and clean data for analysis
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn as sk
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [48]:
#set working directory
os.chdir("C:/Users/A.Melguizo001/Downloads/Immig, Wages, Latinos")

#preliminary coding
    #defining data frames
df_1 = pd.read_csv('gaston_df_v2.csv')
df_hisp = df_1.copy()

## New var construction

In [ ]:
#construcing a new var that measures roughly whether a person got their bachelor's or higher in the US
df_hisp['US_educ'] = 0

hispan_educ_cases = [
    ((df_hisp['educ_clean'] >= 3) & (df_hisp['AGEATIMMIG'] <= 17), 1)
]

df_hisp['US_educ'] = (
    pd.Series(np.nan, index = df_hisp.index)
    .case_when(hispan_educ_cases)
    .fillna(0)
    .astype(int)
)

### interaction vars

In [ ]:
# df_hisp['sexXhisp'] = 0

## Data Cleaning

In [50]:
#dropping hisp_clean to re-categorize
df_hisp = df_hisp.drop(columns= ['hisp_clean'])

#re-categorizing hispanic category
hispan_cases = [
    (df_hisp['HISPAN'] == 0, 0),
    ((df_hisp['HISPAN'] >= 100) & (df_hisp['HISPAN'] <= 109), 1),
    (df_hisp['HISPAN'] == 200, 2),
    (df_hisp['HISPAN'] == 300, 3), 
    (df_hisp['HISPAN'] == 400, 4),
    (df_hisp['HISPAN'] == 500, 5),
    ((df_hisp['HISPAN'] == 610) | (df_hisp['HISPAN'] == 611), 6),
    (df_hisp['HISPAN'] == 612, 7),
    (df_hisp['HISPAN'] == 600, 8),
    ((df_hisp['HISPAN'].isna()) | (df_hisp['HISPAN'] == 902), 9)
]

df_hisp['hisp_clean'] = (
    pd.Series(np.nan, index = df_hisp.index)
    .case_when(hispan_cases)
    .fillna(9)
    .astype(int)
)

df_hisp

,YEAR,REGION,STATEFIP,AGE,SEX,RACE,BPL,YRIMMIG,CITIZEN,NATIVITY,HISPAN,EMPSTAT,UHRSWORKT,EDUC,INCTOT,INCWAGE,YEAROFBIRTH,yrimmig_clean,AGEATIMMIG,years_since_immig,race_clean,bpl_binary,bpl_detail,nativity_clean,educ_clean,region_clean,sex_clean,citizen_clean,empstat_clean,incwage_clean,inctot_clean,US_educ,hisp_clean
0,2000,11,23,51,2,100,45300,1,3,5,0,10,48,125,78644,44600,1949,1949,0,51,1,1,4.0,5.0,4,1,1,1,1,44600.0,78644.000,1,0
1,2000,11,23,41,1,100,41400,3,4,5,0,10,40,81,37525,37500,1959,1962,3,38,1,1,4.0,5.0,2,1,0,2,1,37500.0,37525.000,0,0
2,2000,11,23,59,1,100,41100,3,4,5,0,10,40,40,23025,23000,1941,1962,21,38,1,1,4.0,5.0,0,1,0,2,1,23000.0,23025.000,0,0
3,2000,11,23,55,2,100,15000,3,4,5,0,10,12,73,5009,4860,1945,1962,17,38,1,1,0.0,5.0,1,1,1,2,1,4860.0,5009.000,0,0
4,2000,11,23,59,1,100,11000,5,2,5,200,10,40,91,44000,44000,1941,1972,31,28,1,0,0.0,5.0,2,1,0,1,1,44000.0,44000.000,0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
415916,2025,42,15,58,2,651,51500,38,4,5,0,10,30,92,39046,37000,1967,2008,41,17,4,1,5.0,5.0,2,4,1,2,1,19647.0,20733.426,0,0
415917,2025,42,15,61,2,651,51500,7,4,5,0,10,44,73,56000,56000,1964,1980,16,45,4,1,5.0,5.0,1,4,1,2,1,29736.0,29736.000,0,0
415918,2025,42,15,36,2,651,50000,41,5,5,0,10,997,81,16101,16100,1989,2010,21,15,4,1,5.0,5.0,2,4,1,0,1,8549.1,8549.631,0,0
415919,2025,42,15,55,2,651,51500,9,4,5,0,34,999,91,35801,0,1970,1984,14,41,4,1,5.0,5.0,2,4,1,2,3,0.0,19010.331,0,0


In [51]:
#filter to only include:

#respondents who are: hispanic, born outside of the US and territories, have data for AGEATIMMIG, and have a non-0 wage.
df_hisp = df_hisp[
    (df_hisp['hisp_clean'] > 0) &
    (df_hisp['bpl_binary'] == 1) &
    (df_hisp['AGEATIMMIG'].notna()) &
    (df_hisp['incwage_clean'] > 0)
]

#making log wages for regression
df_hisp['log_wage'] = np.log(df_hisp['incwage_clean'])
df_hisp

# Massachusetts filter
df_hisp_MA = df_hisp[
    (df_hisp['STATEFIP'] == 25)
]

## Data vis beginning

In [52]:
#code for possible data visualization on demographics

#making a new data frame that counts the number of respondents in each hispanic category

# df_hisp_MA_sum = (
#     df_hisp_MA
#     .groupby(['hisp_clean'])
#     .size()
#     .reset_index(name='count')
# )

# #renaming hisp_clean categories to be strings
# mapping = {
#     1: 'Mexican',
#     2: 'Puerto Rican',
#     3: 'Cuban',
#     4: 'Dominican',
#     5: 'Salvadoran',
#     6: 'Central American (except Salvadoran)',
#     7: 'South American',
#     8: 'Other Hispanic',
# }

# df_hisp_MA_sum['hisp_clean'] = df_hisp_MA_sum['hisp_clean'].map(mapping)

# df_hisp_MA_sum

## Regression modeling using statsmodels

### MA Hispanic Regression

In [53]:
#full regression on MA hispanic data
# model_MA = smf.ols(formula = 'log_wage ~ years_since_immig + C(hisp_clean) + sex_clean + C(educ_clean) + C(race_clean) + UHRSWORKT + C(EMPSTAT) + C(STATEFIP)', data = df_hisp_MA)
# results_MA = model_MA.fit()

# print(results_MA.summary())

### Overall US Hispanic regression

In [60]:
#regression on US hispanic data
model = smf.ols(formula = """
log_wage ~ 
years_since_immig + 
sex_clean + 
UHRSWORKT + 
US_educ +
C(CITIZEN) +
C(hisp_clean) + 
C(educ_clean) + 
C(race_clean) + 
C(EMPSTAT) + 
C(STATEFIP)
"""
, data = df_hisp)
results = model.fit()

print(results.summary())

                            OLS Regression Results                            
Dep. Variable:               log_wage   R-squared:                       0.245
Model:                            OLS   Adj. R-squared:                  0.245
Method:                 Least Squares   F-statistic:                     671.3
Date:                Wed, 19 Aug 2026   Prob (F-statistic):               0.00
Time:                        15:21:57   Log-Likelihood:            -1.8687e+05
No. Observations:              165152   AIC:                         3.739e+05
Df Residuals:                  165071   BIC:                         3.747e+05
Df Model:                          80                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             10.0003      0